# FBXL4

Cohort: `FBXL4`, Phenopacket Store v0.1.27 (n = 95).

All individuals were annotated with encephalomyopathic mitochondrial DNA depletion syndrome 13 (MTDPS13), providing a single-disease cohort associated with pathogenic variation in `FBXL4`. 

In [1]:
from contextlib import contextmanager
from pathlib import Path
from zipfile import ZipFile
import os
import math
import numpy as np
import pandas as pd

from ppktstore.model import PhenopacketStore

from phenocor import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
)

PHENOPACKET_STORE_VERSION = "0.1.27"
HPO_RELEASE = "v2026-06-23"
FDR_THRESHOLD = 0.05
ASSOCIATION_MIN_EFFECTIVE_N = 20

# Set phenocor_PROJECT_ROOT or edit this default if the repository is elsewhere.
PROJECT_ROOT = Path(
    os.environ.get("phenocor_PROJECT_ROOT", "..")
).expanduser().resolve()
STORE_ZIP = PROJECT_ROOT / "data" / "all_phenopackets.zip"

COHORT = "FBXL4"
CASE_DIR = PROJECT_ROOT / "results" / COHORT

if not STORE_ZIP.is_file():
    raise FileNotFoundError(f"Phenopacket Store ZIP not found: {STORE_ZIP}")


@contextmanager
def open_local_phenopacket_store(zip_path):
    with ZipFile(zip_path, mode="r") as zip_file:
        store = PhenopacketStore.from_release_zip(
            zip_file=zip_file,
            strategy="lazy",
        )
        yield store

In [2]:
with open_local_phenopacket_store(STORE_ZIP) as store:
    phenopackets = list(store.iter_cohort_phenopackets(COHORT))

print(f"Loaded {len(phenopackets)} phenopackets")

dataset = PhenotypeDatasetBuilder(
    phenopackets,
    hpo_release=HPO_RELEASE,
).build()

dataset.get_diseases().value_counts()

Loaded 95 phenopackets


diseases
[OMIM:615471]    95
Name: count, dtype: int64

## Pairwise correlation

In [3]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

t = results.correlation_results
print(f"tested pairs: {len(t)} | significant: {int((t['adj_p_value'] < 0.05).sum())}")
t.head()

Calculating pairwise correlations:   0%|          | 0/1314 [00:00<?, ?it/s]

Calculating pairwise correlations: 100%|██████████| 1314/1314 [00:01<00:00, 1019.30it/s]


tested pairs: 1117 | significant: 20


,HPO_A,HPO_B,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,HPO_A_label,HPO_B_label,n_pmids,pmids
0,HP:0000648,HP:0007663,1.000000,8.637436e-09,0.000010,44,0,0,7,51,Optic atrophy,Reduced visual acuity,3,23993194;25868664;28940506
1,HP:0000369,HP:0000431,0.715269,3.214973e-07,0.000102,49,2,4,10,65,Low-set ears,Wide nasal bridge,5,23993194;25868664;27099744;27290639;28940506
2,HP:0000319,HP:0012745,0.800228,2.885548e-07,0.000102,58,1,2,7,68,Smooth philtrum,Short palpebral fissure,8,23993194;25868664;26404457;27099744;27182039;2...
3,HP:0001508,HP:0004322,0.718892,3.663681e-07,0.000102,19,2,5,23,49,Failure to thrive,Short stature,4,23993194;25868664;27743463;28940506
4,HP:0000252,HP:0004322,0.714525,4.902156e-07,0.000110,22,4,3,20,49,Microcephaly,Short stature,4,23993194;25868664;27743463;28940506


In [6]:
results.plot_correlation_heatmap_with_significance(
    title="Cohort FBXL4",
)

In [7]:
MAIN_TEXT_MIN_PHI = 0.60


SUPERSCRIPT_TRANSLATION = str.maketrans(
    "0123456789-",
    "⁰¹²³⁴⁵⁶⁷⁸⁹⁻",
)

def format_p_value(value: float) -> str:
    """Format p values using Unicode superscripts for manuscript tables."""
    if value >= 0.001:
        return f"{value:.3f}"

    exponent = math.floor(math.log10(value))
    coefficient = value / (10 ** exponent)
    exponent_text = str(exponent).translate(SUPERSCRIPT_TRANSLATION)

    return f"{coefficient:.2f}×10{exponent_text}"


def phenotype_name(row: pd.Series, term: str) -> str:
    """Combine an HPO label and identifier into one table field."""
    return f"{row[f'HPO_{term}_label']} ({row[f'HPO_{term}']})"


def format_p_values(row: pd.Series) -> str:
    """
    First line: BH FDR-adjusted P value.
    Second line in parentheses: Fisher's exact P value.
    """
    return (
        f"{format_p_value(row['adj_p_value'])}\n"
        f"({format_p_value(row['p_value'])})"
    )


# All FDR-significant phenotype-state associations.
all_significant_associations = (
    t.loc[t["adj_p_value"] < FDR_THRESHOLD]
    .copy()
    .sort_values(
        by=["adj_p_value", "correlation"],
        ascending=[True, False],
    )
)

# Main-text subset: FDR-significant associations with phi > 0.60.
selected_associations = (
    all_significant_associations.loc[
        all_significant_associations["correlation"] > MAIN_TEXT_MIN_PHI
    ]
    .copy()
    .sort_values(
        by=["correlation", "adj_p_value"],
        ascending=[False, True],
    )
)


def prepare_association_table(
    association_df: pd.DataFrame,
    include_publication_count: bool,
) -> pd.DataFrame:
    """Create a manuscript-facing association table."""
    table = pd.DataFrame(
        {
            "Phenotype A": association_df.apply(
                phenotype_name,
                axis=1,
                term="A",
            ),
            "Phenotype B": association_df.apply(
                phenotype_name,
                axis=1,
                term="B",
            ),
            "Phi": association_df["correlation"].map(
                lambda value: f"{value:.2f}"
            ),
            "Total": association_df["n_individuals"].astype(int),
            "State counts (O/O; O/E; E/O; E/E)": (
                association_df["n(A:O/B:O)"].astype(int).astype(str)
                + "; "
                + association_df["n(A:O/B:E)"].astype(int).astype(str)
                + "; "
                + association_df["n(A:E/B:O)"].astype(int).astype(str)
                + "; "
                + association_df["n(A:E/B:E)"].astype(int).astype(str)
            ),
            "FDR-adjusted P (Fisher's exact P)": association_df.apply(
                format_p_values,
                axis=1,
            ),
        }
    )

    if include_publication_count:
        table["PMIDs (n)"] = association_df[
            "n_pmids"
        ].astype(int)

    return table.reset_index(drop=True)


# Main-text Table 2: selected high-magnitude associations.
main_table_2 = prepare_association_table(
    selected_associations,
    include_publication_count=True,
)

# Supplementary Table: all FDR-significant associations, including PMID count.
supplementary_table = prepare_association_table(
    all_significant_associations,
    include_publication_count=True,
)


# Save publication-ready tables without pandas row indices.
main_table_2.to_csv(
    CASE_DIR / "table2_selected_associations_FBXL4.csv",
    index=False,
)

supplementary_table.to_csv(
    CASE_DIR / "supplementary_all_significant_associations_FBXL4.csv",
    index=False,
)


print(
    f"FDR-significant pairs: {len(all_significant_associations)}\n"
    f"Main-text pairs with phi > {MAIN_TEXT_MIN_PHI:.2f}: "
    f"{len(main_table_2)}"
)

display(main_table_2)

FDR-significant pairs: 20
Main-text pairs with phi > 0.60: 9


,Phenotype A,Phenotype B,Phi,Total,State counts (O/O; O/E; E/O; E/E),FDR-adjusted P (Fisher's exact P),PMIDs (n)
0,Optic atrophy (HP:0000648),Reduced visual acuity (HP:0007663),1.00,51,7; 0; 0; 44,9.65×10⁻⁶\n(8.64×10⁻⁹),3
1,Smooth philtrum (HP:0000319),Short palpebral fissure (HP:0012745),0.80,68,7; 2; 1; 58,1.02×10⁻⁴\n(2.89×10⁻⁷),8
2,Failure to thrive (HP:0001508),Short stature (HP:0004322),0.72,49,23; 5; 2; 19,1.02×10⁻⁴\n(3.66×10⁻⁷),4
3,Low-set ears (HP:0000369),Wide nasal bridge (HP:0000431),0.72,65,10; 4; 2; 49,1.02×10⁻⁴\n(3.21×10⁻⁷),5
4,Microcephaly (HP:0000252),Short stature (HP:0004322),0.71,49,20; 3; 4; 22,1.10×10⁻⁴\n(4.90×10⁻⁷),4
5,Nystagmus (HP:0000639),Short palpebral fissure (HP:0012745),0.68,48,5; 3; 1; 39,0.018\n(1.85×10⁻⁴),5
6,Decreased activity of mitochondrial complex II...,Decreased activity of mitochondrial complex II...,0.64,35,10; 2; 4; 19,0.022\n(2.61×10⁻⁴),4
7,Wide nasal bridge (HP:0000431),Thick eyebrow (HP:0000574),0.61,67,8; 6; 2; 51,0.003\n(1.71×10⁻⁵),7
8,Smooth philtrum (HP:0000319),Prominent forehead (HP:0011220),0.60,67,7; 1; 7; 52,0.005\n(2.83×10⁻⁵),7


## Provenance audit

In [7]:
hpo_matrix = dataset.hpo_data.matrix.copy()

pmid_by_individual = (
    hpo_matrix.index.to_series()
    .str.extract(r"^PMID_(\d+)(?:_|$)", expand=False)
)

if pmid_by_individual.isna().any():
    unmatched_ids = pmid_by_individual.index[
        pmid_by_individual.isna()
    ].tolist()
    raise ValueError(
        "Could not extract a PMID from these individual IDs: "
        + str(unmatched_ids[:10])
    )

pmid_by_individual.name = "pmid"

cohort_source_summary = (
    pmid_by_individual
    .value_counts()
    .rename_axis("PMID")
    .reset_index(name="n_individuals")
)

cohort_source_summary["proportion_of_cohort"] = (
    cohort_source_summary["n_individuals"]
    / len(hpo_matrix)
)

cohort_source_summary

,PMID,n_individuals,proportion_of_cohort
0,28940506,37,0.389474
1,25868664,21,0.221053
2,23993194,12,0.126316
3,27743463,10,0.105263
4,27290639,3,0.031579
5,31969900,2,0.021053
6,31442532,2,0.021053
7,32559514,1,0.010526
8,27099744,1,0.010526
9,30804983,1,0.010526


In [35]:
def audit_association_provenance(
    association_df: pd.DataFrame,
    hpo_matrix: pd.DataFrame,
    pmid_by_individual: pd.Series,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Create provenance summaries for phenotype-state association pairs.

    The audit uses the PMID encoded in each individual_id.
    It does not change the association test or its FDR correction.
    """
    summary_rows = []
    contribution_rows = []

    for _, row in association_df.iterrows():
        hpo_a = row["HPO_A"]
        hpo_b = row["HPO_B"]

        # Keep only individuals with an informative state for both terms.
        pair_data = hpo_matrix[[hpo_a, hpo_b]].dropna().copy()
        pair_pmids = pmid_by_individual.loc[pair_data.index]

        if pair_pmids.isna().any():
            raise AssertionError(
                f"Missing PMID for one or more informative individuals in "
                f"{row['HPO_A_label']} and {row['HPO_B_label']}."
            )

        # Matrix encoding: 1 = observed; 0 = excluded.
        n_11 = int(
            ((pair_data[hpo_a] == 1) & (pair_data[hpo_b] == 1)).sum()
        )
        n_10 = int(
            ((pair_data[hpo_a] == 1) & (pair_data[hpo_b] == 0)).sum()
        )
        n_01 = int(
            ((pair_data[hpo_a] == 0) & (pair_data[hpo_b] == 1)).sum()
        )
        n_00 = int(
            ((pair_data[hpo_a] == 0) & (pair_data[hpo_b] == 0)).sum()
        )

        if n_11 + n_10 + n_01 + n_00 != len(pair_data):
            raise AssertionError(
                "Four-state counts do not sum to the pair-specific "
                "effective sample size."
            )

        # Verify that the audit reproduces the analyzer's result table.
        expected_counts = {
            "n(A:E/B:E)": n_00,
            "n(A:E/B:O)": n_01,
            "n(A:O/B:E)": n_10,
            "n(A:O/B:O)": n_11,
        }

        for result_column, audit_count in expected_counts.items():
            if int(row[result_column]) != audit_count:
                raise AssertionError(
                    f"Count mismatch for {row['HPO_A_label']} and "
                    f"{row['HPO_B_label']}: {result_column} is "
                    f"{row[result_column]} in the result table but "
                    f"{audit_count} in the matrix."
                )

        cell_labels = pd.Series(
            np.select(
                [
                    (pair_data[hpo_a] == 1) & (pair_data[hpo_b] == 1),
                    (pair_data[hpo_a] == 1) & (pair_data[hpo_b] == 0),
                    (pair_data[hpo_a] == 0) & (pair_data[hpo_b] == 1),
                    (pair_data[hpo_a] == 0) & (pair_data[hpo_b] == 0),
                ],
                [
                    "both_observed",
                    "A_observed_B_excluded",
                    "A_excluded_B_observed",
                    "both_excluded",
                ],
                default="unclassified",
            ),
            index=pair_data.index,
            name="state_cell",
        )

        if cell_labels.eq("unclassified").any():
            raise AssertionError(
                "Unexpected phenotype state in pair_data; expected only 0 or 1."
            )

        informative_source_counts = pair_pmids.value_counts()
        both_observed_pmids = pair_pmids.loc[
            cell_labels.eq("both_observed")
        ]
        both_observed_source_counts = both_observed_pmids.value_counts()

        summary_rows.append(
            {
                "Phenotype A": row["HPO_A_label"],
                "HPO A": hpo_a,
                "Phenotype B": row["HPO_B_label"],
                "HPO B": hpo_b,
                "Phi": row["correlation"],
                "FDR-adjusted P": row["adj_p_value"],
                "N_eff": len(pair_data),
                "n_both_observed": n_11,
                "n_A_observed_B_excluded": n_10,
                "n_A_excluded_B_observed": n_01,
                "n_both_excluded": n_00,
                "n_PMIDs_informative": informative_source_counts.size,
                "PMIDs_informative": ";".join(
                    map(str, informative_source_counts.index.tolist())
                ),
                "n_PMIDs_both_observed": both_observed_source_counts.size,
                "PMIDs_both_observed": ";".join(
                    map(str, both_observed_source_counts.index.tolist())
                ),
                "dominant_PMID_both_observed": (
                    both_observed_source_counts.index[0]
                    if not both_observed_source_counts.empty
                    else pd.NA
                ),
                "dominant_PMID_share_both_observed": (
                    both_observed_source_counts.iloc[0] / n_11
                    if n_11 > 0
                    else pd.NA
                ),
            }
        )

        # Retain individual-level data for internal auditing only.
        for individual_id, state_cell in cell_labels.items():
            contribution_rows.append(
                {
                    "Phenotype A": row["HPO_A_label"],
                    "HPO A": hpo_a,
                    "Phenotype B": row["HPO_B_label"],
                    "HPO B": hpo_b,
                    "individual_id": individual_id,
                    "PMID": pmid_by_individual.loc[individual_id],
                    "state_cell": state_cell,
                }
            )

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(contribution_rows),
    )


# Run the pair-level provenance audit.
fbxl4_provenance_audit, fbxl4_source_contributions = (
    audit_association_provenance(
        association_df=all_significant_associations,
        hpo_matrix=hpo_matrix,
        pmid_by_individual=pmid_by_individual,
    )
)


# Create a compact supplementary-data table:
# one row per phenotype pair × PMID, with four-state counts.
PAIR_COLUMNS = [
    "Phenotype A",
    "HPO A",
    "Phenotype B",
    "HPO B",
]

STATE_COLUMNS = [
    "both_observed",
    "A_observed_B_excluded",
    "A_excluded_B_observed",
    "both_excluded",
]

STATE_RENAME = {
    "both_observed": "n_both_observed",
    "A_observed_B_excluded": "n_A_observed_B_excluded",
    "A_excluded_B_observed": "n_A_excluded_B_observed",
    "both_excluded": "n_both_excluded",
}

COUNT_COLUMNS = list(STATE_RENAME.values())

fbxl4_source_state_counts = (
    fbxl4_source_contributions
    .groupby(PAIR_COLUMNS + ["PMID", "state_cell"], dropna=False)
    .size()
    .unstack("state_cell", fill_value=0)
    .reindex(columns=STATE_COLUMNS, fill_value=0)
    .rename(columns=STATE_RENAME)
    .reset_index()
)

fbxl4_source_state_counts["N_eff_within_PMID"] = (
    fbxl4_source_state_counts[COUNT_COLUMNS].sum(axis=1)
)

fbxl4_source_state_counts = (
    fbxl4_source_state_counts
    .sort_values(PAIR_COLUMNS + ["PMID"])
    .reset_index(drop=True)
)


# Verify that summing PMID-level counts reproduces cohort-level counts.
source_totals = (
    fbxl4_source_state_counts
    .groupby(PAIR_COLUMNS)[COUNT_COLUMNS]
    .sum()
    .sort_index()
)

audit_totals = (
    fbxl4_provenance_audit
    .set_index(PAIR_COLUMNS)[COUNT_COLUMNS]
    .sort_index()
)

if not source_totals.equals(audit_totals):
    raise AssertionError(
        "PMID-level state counts do not reproduce cohort-level audit counts."
    )

print(
    "PMID-level four-state counts successfully reproduce "
    "the cohort-level audit."
)


# Export the two manuscript-facing provenance outputs.
fbxl4_provenance_audit.to_csv(
    QC_DIR / "FBXL4_significant_association_provenance_audit.csv",
    index=False,
)

fbxl4_source_state_counts.to_csv(
    QC_DIR / "FBXL4_significant_association_source_state_counts.csv",
    index=False,
)
display(fbxl4_provenance_audit)
display(fbxl4_source_state_counts)

PMID-level four-state counts successfully reproduce the cohort-level audit.


,Phenotype A,HPO A,Phenotype B,HPO B,Phi,FDR-adjusted P,N_eff,n_both_observed,n_A_observed_B_excluded,n_A_excluded_B_observed,n_both_excluded,n_PMIDs_informative,PMIDs_informative,n_PMIDs_both_observed,PMIDs_both_observed,dominant_PMID_both_observed,dominant_PMID_share_both_observed
0,Optic atrophy,HP:0000648,Reduced visual acuity,HP:0007663,1.000000,0.000010,51,7,0,0,44,3,28940506;25868664;23993194,3,28940506;25868664;23993194,28940506,0.714286
1,Smooth philtrum,HP:0000319,Short palpebral fissure,HP:0012745,0.800228,0.000102,68,7,2,1,58,8,28940506;25868664;23993194;27099744;27182039;3...,5,25868664;27182039;28940506;34602956;26404457,25868664,0.428571
2,Failure to thrive,HP:0001508,Short stature,HP:0004322,0.718892,0.000102,49,23,5,2,19,4,28940506;27743463;25868664;23993194,4,28940506;25868664;27743463;23993194,28940506,0.782609
3,Low-set ears,HP:0000369,Wide nasal bridge,HP:0000431,0.715269,0.000102,65,10,4,2,49,5,28940506;25868664;23993194;27099744;27290639,2,25868664;28940506,25868664,0.700000
4,Microcephaly,HP:0000252,Short stature,HP:0004322,0.714525,0.000110,49,20,3,4,22,4,28940506;27743463;23993194;25868664,3,28940506;25868664;27743463,28940506,0.800000
5,Wide nasal bridge,HP:0000431,Thick eyebrow,HP:0000574,0.608914,0.003187,67,8,6,2,51,7,28940506;25868664;23993194;27099744;27182039;3...,4,25868664;27182039;28940506;34602956,25868664,0.625000
6,Smooth philtrum,HP:0000319,Prominent forehead,HP:0011220,0.603247,0.004524,67,7,1,7,52,7,28940506;25868664;23993194;27099744;27182039;3...,4,25868664;28940506;27182039;34602956,25868664,0.428571
7,Long philtrum,HP:0000343,Low-set ears,HP:0000369,0.533578,0.018270,66,8,3,7,48,6,28940506;25868664;23993194;27099744;32779419;2...,3,28940506;25868664;32779419,28940506,0.500000
8,Nystagmus,HP:0000639,Short palpebral fissure,HP:0012745,0.676123,0.018336,48,5,3,1,39,5,28940506;25868664;23993194;27182039;34602956,4,28940506;25868664;27182039;34602956,28940506,0.400000
9,Prominent forehead,HP:0011220,Short palpebral fissure,HP:0012745,0.544562,0.018336,67,6,8,1,52,7,28940506;25868664;23993194;27099744;27182039;3...,4,25868664;28940506;27182039;34602956,25868664,0.500000


state_cell,Phenotype A,HPO A,Phenotype B,HPO B,PMID,n_both_observed,n_A_observed_B_excluded,n_A_excluded_B_observed,n_both_excluded,N_eff_within_PMID
0,Decreased activity of mitochondrial complex II,HP:0008314,Decreased activity of mitochondrial complex III,HP:0011924,23993194,1,0,1,5,7
1,Decreased activity of mitochondrial complex II,HP:0008314,Decreased activity of mitochondrial complex III,HP:0011924,25868664,4,1,1,9,15
2,Decreased activity of mitochondrial complex II,HP:0008314,Decreased activity of mitochondrial complex III,HP:0011924,27290639,1,0,0,0,1
3,Decreased activity of mitochondrial complex II,HP:0008314,Decreased activity of mitochondrial complex III,HP:0011924,28940506,4,1,2,5,12
4,Epicanthus,HP:0000286,Long eyelashes,HP:0000527,23993194,1,0,0,8,9
...,...,...,...,...,...,...,...,...,...,...
110,Wide nasal bridge,HP:0000431,Thick eyebrow,HP:0000574,27099744,0,0,0,1,1
111,Wide nasal bridge,HP:0000431,Thick eyebrow,HP:0000574,27182039,1,0,0,0,1
112,Wide nasal bridge,HP:0000431,Thick eyebrow,HP:0000574,27290639,0,0,0,1,1
113,Wide nasal bridge,HP:0000431,Thick eyebrow,HP:0000574,28940506,1,3,1,32,37
